# Feature Track 0b: Document Ingestion and Chunking

---

This notebook explores how raw documents are converted into chunks that the RAG pipeline can retrieve and reason over. It is a companion to `feature0a_baseline_rag.ipynb`, which covers the full pipeline end-to-end.

**What this notebook covers:**
1. Document size limits and fallback behaviour
2. PDF parsing: comparing two conversion engines
3. Chunking strategies: header-based, fixed-size, and paragraph-aware
4. Chunk size distributions and the embedding token limit
5. Tables, images, and Excel files
6. How chunking choices affect retrieval quality

**Prerequisite:** `conversational-toolkit` and `backend` must be installed in editable mode.


---

## Setup


In [17]:
import re
import warnings

import pymupdf4llm  # type: ignore[import-untyped]
from markitdown import MarkItDown  # type: ignore[import-untyped]
from docling.document_converter import DocumentConverter  # type: ignore[import-untyped]

from conversational_toolkit.chunking.excel_chunker import ExcelChunker

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import DATA_DIR, MAX_FILE_SIZE_MB
from sme_kt_zh_collaboration_rag.feature0_ingestion import (
    analyze_chunks,
    char_histogram,
    compare_strategies,
    fixed_size_chunks,
    header_based_chunks,
    paragraph_aware_chunks,
    print_comparison_table,
)

warnings.filterwarnings("ignore")

# Pick a representative PDF from the corpus for demonstrations
sample_pdf = str(DATA_DIR / "EPD_pallet_relicyc_logypal1.pdf")
print(f"Sample file: {sample_pdf}")
print(f"Data directory: {DATA_DIR}")
print(f"File size limit: {MAX_FILE_SIZE_MB} MB")

Sample file: /Users/pkoerner/Desktop/Kanton_Zurich/sme-kt-zh-collaboration-rag/data/EPD_pallet_relicyc_logypal1.pdf
Data directory: /Users/pkoerner/Desktop/Kanton_Zurich/sme-kt-zh-collaboration-rag/data
File size limit: 20 MB


---

## 1. Document Size Limits

Not every PDF can be parsed safely. Very large files (image-heavy catalogues, scanned books) can exhaust memory or take minutes to convert. The ingestion pipeline checks file size before attempting to parse and skips files above the limit.


In [2]:
# Show file sizes in the data directory and flag files above the limit
print(f"{'File':<55}  {'Size (MB)':>9}  {'Status'}")
print("-" * 80)
for f in sorted(DATA_DIR.iterdir()):
    if not f.is_file():
        continue
    size_mb = f.stat().st_size / (1024 * 1024)
    status = "SKIP (too large)" if size_mb > MAX_FILE_SIZE_MB else "ok"
    flag = "  <--" if size_mb > MAX_FILE_SIZE_MB else ""
    print(f"  {f.name:<53}  {size_mb:>9.1f}  {status}{flag}")

File                                                     Size (MB)  Status
--------------------------------------------------------------------------------
  .DS_Store                                                    0.0  ok
  ART_internal_procurement_policy.pdf                          0.1  ok
  ART_logylight_incomplete_datasheet.pdf                       0.1  ok
  ART_product_catalog.pdf                                      0.1  ok
  ART_product_overview.xlsx                                    0.0  ok
  ART_relicyc_logypal1_datasheet_2021.pdf                      0.1  ok
  ART_response_inquiry_frische_felder.md                       0.0  ok
  ART_supplier_brochure_CPR_wood_pallet.pdf                    0.1  ok
  ART_supplier_brochure_tesa_ECO.pdf                           0.1  ok
  EPD_cardboard_grupak_corrugated.pdf                          4.6  ok
  EPD_cardboard_redbox_cartonpallet.pdf                        0.8  ok
  EPD_pallet_CPR_noe.pdf                                       

---

## 2. PDF Parser Comparison: pymupdf4llm vs MarkItDown vs Docling

Before chunking, a PDF must be converted to plain text. Three engines are available:

| Engine | Approach | Strengths | Weaknesses |
|--------|----------|-----------|------------|
| `pymupdf4llm` | Low-level PDF parsing, extracts text directly | Fast, good heading detection, reliable layout | Struggles with complex multi-column layouts |
| `markitdown` | Converts via an intermediate format | Handles more file types (DOCX, PPTX) | Slower, sometimes loses structure |
| `docling` | AI-assisted layout analysis with ML models | Best table extraction, handles complex layouts, reading order | Slow (first run downloads models), heavy dependency |

The choice of parser affects which headings are detected, whether tables survive as Markdown, how much text is dropped from figures or sidebars, and overall parsing speed.

**Docling note:** On first use, Docling downloads its ML models. Subsequent runs reuse the cached models and are faster.

In [5]:
import time

# Parse the same file with all three engines and compare output length
t0 = time.time()
md_pymupdf = pymupdf4llm.to_markdown(sample_pdf)
t_pymupdf = time.time() - t0
t0 = time.time()
md_markitdown = str(MarkItDown().convert(sample_pdf).text_content)
t_markitdown = time.time() - t0
t0 = time.time()
md_docling = DocumentConverter().convert(sample_pdf).document.export_to_markdown()
t_docling = time.time() - t0

print(f"{'Engine':<15}  {'Chars':>8}  {'Lines':>7}  {'Time':>8}")
print("─" * 45)
print(
    f"{'pymupdf4llm':<15}  {len(md_pymupdf):>8,}  {len(md_pymupdf.splitlines()):>7,}  {t_pymupdf:>7.1f}s"
)
print(
    f"{'markitdown':<15}  {len(md_markitdown):>8,}  {len(md_markitdown.splitlines()):>7,}  {t_markitdown:>7.1f}s"
)
print(
    f"{'docling':<15}  {len(md_docling):>8,}  {len(md_docling.splitlines()):>7,}  {t_docling:>7.1f}s"
)

Engine              Chars    Lines      Time
─────────────────────────────────────────────
pymupdf4llm        21,826      813      1.3s
markitdown         19,918    1,365      0.2s
docling            24,479      344     10.1s


In [6]:
def extract_headings(md_text):
    return [
        line.strip() for line in md_text.splitlines() if re.match(r"^#{1,4} ", line)
    ]


headings_pymupdf = extract_headings(md_pymupdf)
headings_markitdown = extract_headings(md_markitdown)
headings_docling = extract_headings(md_docling)

for label, headings in [
    ("pymupdf4llm", headings_pymupdf),
    ("markitdown", headings_markitdown),
    ("docling", headings_docling),
]:
    print(f"Headings found by {label} ({len(headings)}):")
    for h in headings[:15]:
        print(f"  {h}")
    print()

Headings found by pymupdf4llm (17):
  # PROGRAMME INFORMATION
  ### Accountabilities for PCR, LCA and independent, third-party verification PRODUCT CATEGORY RULES (PCR)
  ### LIFE CYCLE ASSESSMENT (LCA)
  ### THIRD-PARTY VERIFICATION
  # 40 ANNI DI INNOVAZIONE SOSTENIBILE
  # 40 YEARS OF SUSTAINABLE INNOVATION
  # VICINI AI CLIENTI PER CONDIVIDERNE GLI OBIETTIVI
  # CLOSER TO CUSTOMERS THROUGH SHARED GOALS
  # COMPANY INFORMATION
  # PRODUCT INFORMATION
  # THE SYSTEM BOUNDARIES
  # LCA INFORMATION
  # CALCULATION RULES
  # CONTENT DECLARATION
  # ENVIRONMENTAL PERFORMANCE

Headings found by markitdown (0):

Headings found by docling (23):
  ## ENVIRONMENTAL PRODUCT DECLARATION
  ## PROGRAMME INFORMATION
  ## Accountabilities for PCR, LCA and independent, third-party verification
  ## PRODUCT CATEGORY RULES (PCR)
  ## LIFE CYCLE ASSESSMENT (LCA)
  ## THIRD-PARTY VERIFICATION
  ## [ X ] Yes [ ] No
  ## 40 ANNI DI INNOVAZIONE SOSTENIBILE
  ## 40 YEARS OF SUSTAINABLE INNOVATION
  ## VICIN

In [8]:
# Table extraction comparison: count Markdown pipe tables in each parser's output
def count_tables(md_text):
    """Count Markdown pipe-table blocks (heuristic: lines starting with |)."""
    in_table = False
    count = 0
    for line in md_text.splitlines():
        stripped = line.strip()
        if stripped.startswith("|"):
            if not in_table:
                in_table = True
                count += 1
        else:
            in_table = False
    return count


for label, md in [
    ("pymupdf4llm", md_pymupdf),
    ("markitdown", md_markitdown),
    ("docling", md_docling),
]:
    n_tables = count_tables(md)
    print(f"{label:<15}: {n_tables} table(s) detected")

# Show a sample table from docling (if any) -> docling's table extraction is most reliable
table_lines = []
in_table = False
for line in md_docling.splitlines():
    if line.strip().startswith("|"):
        table_lines.append(line)
        in_table = True
    elif in_table:
        break

if table_lines:
    print(f"\nFirst table from docling ({len(table_lines)} rows):")
    for row in table_lines:
        print(f"  {row}")
else:
    print("\nNo pipe tables found in docling output for this file.")

pymupdf4llm    : 7 table(s) detected
markitdown     : 0 table(s) detected
docling        : 7 table(s) detected

First table from docling (10 rows):
  | MODEL                                 | Logypal 1    |
  |---------------------------------------|--------------|
  | Dimensions [mm] (LxWxH)               | 1200x800x138 |
  | Weight [kg]                           | 4,5          |
  | Dynamic load [kg]                     | 800          |
  | Static load [kg]                      | 1600         |
  | Stackable                             | yes          |
  | Number of pallets for standard stack  | 62           |
  | Stack heigh for lorry [mm]            | 2610         |
  | Weight for stack loaded in lorry [kg] | 310          |


---

## 3. Chunking Strategies

Three strategies are available. The right choice depends on the document structure:

| Strategy | How it splits | Best for | Risk |
|----------|--------------|----------|------|
| **Header-based** | One chunk per Markdown heading section | Well-structured PDFs with clear sections | Unstructured docs produce very large or very small chunks |
| **Fixed-size** | Every N characters, with overlap | Uniform chunk sizes, predictable embedding behaviour | May cut mid-sentence or mid-table |
| **Paragraph-aware** | Merges paragraphs until target size | Flowing text, narrative documents | Size varies more than fixed |


In [9]:
# Header-based chunking (default)
chunks_header = header_based_chunks(sample_pdf)
stats_header = analyze_chunks(chunks_header, "header_based")
print("Header-based chunks:")
print(stats_header)
print()
print(char_histogram(chunks_header))

Header-based chunks:
header_based           chunks=  17, avg=  1273, min=  100, max=  4342, >256tok=   7

     100-524    | ████████████████████ 3
     524-948    | ████████████████████████████████████████ 6
     948-1373   | █████████████ 2
    1373-1797   | █████████████ 2
    1797-2221   | ██████ 1
    2221-2645   | ██████ 1
    2645-3069   |  0
    3069-3494   | ██████ 1
    3494-3918   |  0
    3918-4342   | ██████ 1


In [10]:
# Fixed-size chunking
chunks_fixed = fixed_size_chunks(sample_pdf, chunk_size=800, overlap=100)
stats_fixed = analyze_chunks(chunks_fixed, "fixed_size_800")
print("Fixed-size chunks (800 chars, 100 overlap):")
print(stats_fixed)
print()
print(char_histogram(chunks_fixed))

Fixed-size chunks (800 chars, 100 overlap):
fixed_size_800         chunks=  32, avg=   779, min=  126, max=   800, >256tok=   0

     126-193    | █ 1
     193-261    |  0
     261-328    |  0
     328-396    |  0
     396-463    |  0
     463-530    |  0
     530-598    |  0
     598-665    |  0
     665-733    |  0
     733-800    | ████████████████████████████████████████ 31


In [19]:
# Paragraph-aware chunking
chunks_para = paragraph_aware_chunks(sample_pdf, target_chars=500)
stats_para = analyze_chunks(chunks_para, "paragraph_500")
print("Paragraph-aware chunks (target 500 chars):")
print(stats_para)
print()
print(char_histogram(chunks_para))

Paragraph-aware chunks (target 500 chars):
paragraph_500          chunks=  40, avg=   540, min=   27, max=  1824, >256tok=   4

      27-207    | ██████ 5
     207-386    | █ 1
     386-566    | ████████████████████████████████████████ 29
     566-746    | █ 1
     746-926    |  0
     926-1105   | ██ 2
    1105-1285   |  0
    1285-1465   |  0
    1465-1644   |  0
    1644-1824   | ██ 2


In [20]:
# Side-by-side comparison table
results = compare_strategies(sample_pdf)
print_comparison_table(results)

2026-03-01 21:43:51.260 | INFO     | sme_kt_zh_collaboration_rag.feature0_ingestion:compare_strategies:170 - Comparing chunking strategies on: EPD_pallet_relicyc_logypal1.pdf
2026-03-01 21:43:52.757 | INFO     | sme_kt_zh_collaboration_rag.feature0_ingestion:compare_strategies:182 -   header_based           chunks=  17, avg=  1273, min=  100, max=  4342, >256tok=   7
2026-03-01 21:43:54.044 | INFO     | sme_kt_zh_collaboration_rag.feature0_ingestion:compare_strategies:182 -   fixed_size_800         chunks=  32, avg=   779, min=  126, max=   800, >256tok=   0
2026-03-01 21:43:55.324 | INFO     | sme_kt_zh_collaboration_rag.feature0_ingestion:compare_strategies:182 -   paragraph_600          chunks=  33, avg=   654, min=   84, max=  1824, >256tok=   4


Strategy                Chunks  Avg chars    Min     Max  >256tok
-----------------------------------------------------------------
header_based           chunks=  17, avg=  1273, min=  100, max=  4342, >256tok=   7
fixed_size_800         chunks=  32, avg=   779, min=  126, max=   800, >256tok=   0
paragraph_600          chunks=  33, avg=   654, min=   84, max=  1824, >256tok=   4


---

## 4. The Embedding Token Limit

The default embedding model `all-MiniLM-L6-v2` silently truncates input at 256 tokens (~1000 characters). Content beyond that limit is lost before it ever reaches the vector store.

The `>256tok` column in the comparison table above shows how many chunks exceed this limit.

**Options if many chunks are truncated:**
- Switch to a model with a higher token limit (e.g. `text-embedding-3-small` supports 8,191 tokens)
- Reduce chunk size
- Use fixed-size chunking to cap the maximum size


In [21]:
# Show the token distribution across chunking strategies
from sme_kt_zh_collaboration_rag.feature0_ingestion import estimate_tokens

for strategy_name, (chunks, stats) in results.items():
    token_counts = [estimate_tokens(c.content) for c in chunks]
    over = sum(1 for t in token_counts if t > 256)
    pct = 100 * over / len(chunks) if chunks else 0
    print(
        f"{strategy_name:<22}  {over:>4} / {len(chunks)} chunks exceed 256 tokens  ({pct:.0f}%)"
    )

header_based               7 / 17 chunks exceed 256 tokens  (41%)
fixed_size_800             0 / 32 chunks exceed 256 tokens  (0%)
paragraph_600              4 / 33 chunks exceed 256 tokens  (12%)


---

## 5. Tables and Spreadsheets

Tables in PDFs are rendered as Markdown pipe tables after parsing. Quality depends on the table complexity and the parser engine.

Excel files are handled by a separate chunker: one Markdown table per sheet.


In [23]:
# Find chunks that contain Markdown tables in the header-based output
table_chunks = [c for c in chunks_header if c.content.count("|") >= 4]
print(f"Chunks containing tables: {len(table_chunks)}")
if table_chunks:
    print("\nFirst table chunk (first 1000 chars):")
    print(table_chunks[0].content[:1000])

Chunks containing tables: 5

First table chunk (first 1000 chars):
# THE SYSTEM BOUNDARIES

The system boundaries include the entire life cycle of the analyzed

product, in accordance with an LCA approach ”from cradle-to-grave”.

The life cycle modules considered within the system boundaries of

the present study have been grouped into three stages according to

the PCR Packaging. Irrilevant modules will be marked, in the following

table, as ”Module Not Declared, MND”.



Looking at the previous table:

Module A5 is considered as irrilevant because it’s included

in the Manufacturing phase (module A3)

 Module B1 is considered as irrilevant because there is not filling

of the packaging unit with any kind of matter in any physical state

(liquid, solid, or gas)

 Module B3 and B4 are considered as irrilevant because there aren’t

operations necessary to restore a reusable packaging to a functional

state for further reuse

 Module C1 is considered as irrilevant because there aren’t

o

In [ ]:
# Excel files: one chunk per sheet
xlsx_file = str(DATA_DIR / "ART_product_overview.xlsx")
excel_chunks = ExcelChunker().make_chunks(xlsx_file)
print(f"Excel chunks: {len(excel_chunks)}")
for c in excel_chunks:
    print(f"  Sheet: {c.title}")
    print(c.content[:300])

---

## 6. How Chunking Affects Retrieval

The same question can retrieve very different chunks depending on the chunking strategy. Header-based chunks tend to be self-contained, while fixed-size chunks may contain partial context from adjacent sections.

Try the queries below across different chunk sets to see which strategy surfaces the right content.


In [ ]:
# Concrete example: compare what each strategy retrieves for the same query
query_terms = ["GWP", "global warming", "carbon"]

for strategy_name, (chunks, _) in results.items():
    hits = [
        c for c in chunks if any(t.lower() in c.content.lower() for t in query_terms)
    ]
    print(
        f"\n{strategy_name}: {len(hits)} / {len(chunks)} chunks mention {query_terms}"
    )
    if hits:
        print(
            f"  First hit ({len(hits[0].content)} chars): {hits[0].content[:200].strip()!r}"
        )

---

## Summary

### PDF Parsers

| Engine | Speed | Table quality | Heading detection | Best for |
|--------|-------|---------------|-------------------|----------|
| `pymupdf4llm` | Fast | Good | Reliable | Default: fast, lightweight ingestion |
| `markitdown` | Medium | Moderate | Variable | Multi-format corpora (DOCX, PPTX, PDF) |
| `docling` | Slow (ML models) | Excellent | Good | Complex layouts, scanned PDFs, rich tables |

### Chunking Strategies

| Aspect | Header-based | Fixed-size | Paragraph-aware |
|--------|-------------|-----------|-----------------|
| Chunk boundaries | Section headings | Every N chars | Paragraph breaks |
| Size predictability | Low | High | Medium |
| Context preservation | High (one topic per chunk) | Medium (overlap helps) | Medium |
| Truncation risk | High for long sections | Low (tune chunk_size) | Medium |
| Best for | Structured reports, EPDs | Any document | Flowing narrative text |

**Key takeaways:**
- `pymupdf4llm` is the right default: fast and reliable for the structured PDFs in this corpus.
- `docling` is worth the overhead when table fidelity matters (e.g. LCA data tables in EPDs).
- Header-based chunking works well for structured EPD and policy documents. For less structured documents, fixed-size chunking with a size below the 256-token limit is safer.